EDA — Credit Card Clients (Default / Late Payment Risk) by Kasia Mc art

In [58]:

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import random
import plotly.io as pio
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
import plotly.express as px
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)


In [2]:
from sklearn.metrics import precision_recall_curve

In [3]:
import sys
print(sys.executable)

c:\Users\Kasia\AppData\Local\Programs\Python\Python311\python.exe


In [4]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report, confusion_matrix
from sklearn.ensemble import RandomForestClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier


In [5]:
from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)

In [6]:
# ---------------------------------------------------------------------
# loading data
# ---------------------------------------------------------------------
def find_repo_root(start: Path | None = None) -> Path:
    p = (start or Path.cwd()).resolve()
    for parent in [p, *p.parents]:
        if (parent / ".git").exists():
            return parent
    return p

REPO_ROOT = find_repo_root()

In [7]:
# ---------------------------------------------------------------------
# CONFIG
# ---------------------------------------------------------------------
DATA_PATH = REPO_ROOT / "data" / "raw" / "default of credit card clients.xls"
SHEET_NAME = 0
RANDOM_STATE = 42

print("Repo root:", REPO_ROOT)
print("Data path:", DATA_PATH)
print("Exists:", DATA_PATH.exists())


Repo root: F:\Apps\Credit-Risk-Score-ML
Data path: F:\Apps\Credit-Risk-Score-ML\data\raw\default of credit card clients.xls
Exists: True


In [8]:
# ---------------------------------------------------------------------
# LOAD RAW DATA 
# ---------------------------------------------------------------------
df_raw = pd.read_excel(DATA_PATH, sheet_name=0, engine="xlrd", header=1)
print(df_raw.columns.tolist())
df_raw.head()

['ID', 'LIMIT_BAL', 'SEX', 'EDUCATION', 'MARRIAGE', 'AGE', 'PAY_0', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6', 'BILL_AMT1', 'BILL_AMT2', 'BILL_AMT3', 'BILL_AMT4', 'BILL_AMT5', 'BILL_AMT6', 'PAY_AMT1', 'PAY_AMT2', 'PAY_AMT3', 'PAY_AMT4', 'PAY_AMT5', 'PAY_AMT6', 'default payment next month']


,ID,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,PAY_5,PAY_6,BILL_AMT1,BILL_AMT2,BILL_AMT3,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default payment next month
0,1,20000,2,2,1,24,2,2,-1,-1,-2,-2,3913,3102,689,0,0,0,0,689,0,0,0,0,1
1,2,120000,2,2,2,26,-1,2,0,0,0,2,2682,1725,2682,3272,3455,3261,0,1000,1000,1000,0,2000,1
2,3,90000,2,2,2,34,0,0,0,0,0,0,29239,14027,13559,14331,14948,15549,1518,1500,1000,1000,1000,5000,0
3,4,50000,2,2,1,37,0,0,0,0,0,0,46990,48233,49291,28314,28959,29547,2000,2019,1200,1100,1069,1000,0
4,5,50000,1,2,1,57,-1,0,-1,0,0,0,8617,5670,35835,20940,19146,19131,2000,36681,10000,9000,689,679,0


In [9]:
# -----------------------------------------------------------------------------
# HELPERS
# -----------------------------------------------------------------------------
def print_df(df: pd.DataFrame, name: str, n: int = 10):
    print("\n" + "-" * 95)
    print(f"[DATAFRAME] {name} | shape = {df.shape[0]:,} rows × {df.shape[1]:,} cols")
    print("-" * 95)
    print(df.head(n).to_string())

def section(title: str):
    print("\n" + "=" * 95)
    print(title)
    print("=" * 95)

def pct(x):
    return 100 * x


In [10]:
print(df_raw.columns.tolist())


['ID', 'LIMIT_BAL', 'SEX', 'EDUCATION', 'MARRIAGE', 'AGE', 'PAY_0', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6', 'BILL_AMT1', 'BILL_AMT2', 'BILL_AMT3', 'BILL_AMT4', 'BILL_AMT5', 'BILL_AMT6', 'PAY_AMT1', 'PAY_AMT2', 'PAY_AMT3', 'PAY_AMT4', 'PAY_AMT5', 'PAY_AMT6', 'default payment next month']


In [11]:
# =============================================================================
# CLEAN COLUMN NAMES
# =============================================================================
section("Cleaning column names")

df = df_raw.copy()

df = df.rename(columns={"default payment next month": "DEFAULT_NEXT_MONTH"})

if "ID" in df.columns:
    df["ID"] = df["ID"].astype(int)

print_df(df, "cleaned")
print("Target distribution (counts):")
print(df["DEFAULT_NEXT_MONTH"].value_counts(dropna=False).to_string())
print("\nTarget distribution (rate %):")
print((df["DEFAULT_NEXT_MONTH"].value_counts(normalize=True) * 100).round(2).to_string())



Cleaning column names

-----------------------------------------------------------------------------------------------
[DATAFRAME] cleaned | shape = 30,000 rows × 25 cols
-----------------------------------------------------------------------------------------------
   ID  LIMIT_BAL  SEX  EDUCATION  MARRIAGE  AGE  PAY_0  PAY_2  PAY_3  PAY_4  PAY_5  PAY_6  BILL_AMT1  BILL_AMT2  BILL_AMT3  BILL_AMT4  BILL_AMT5  BILL_AMT6  PAY_AMT1  PAY_AMT2  PAY_AMT3  PAY_AMT4  PAY_AMT5  PAY_AMT6  DEFAULT_NEXT_MONTH
0   1      20000    2          2         1   24      2      2     -1     -1     -2     -2       3913       3102        689          0          0          0         0       689         0         0         0         0                   1
1   2     120000    2          2         2   26     -1      2      0      0      0      2       2682       1725       2682       3272       3455       3261         0      1000      1000      1000         0      2000                   1
2   3      90000    2   

## Data dictionary (high level)

**Target**
- `DEFAULT_NEXT_MONTH` — 1 = default next month, 0 = not default

**Static customer profile**
- `LIMIT_BAL` — credit limit
- `SEX`, `EDUCATION`, `MARRIAGE`, `AGE`

**Repayment status (behaviour)**
- `PAY_0`, `PAY_2`, `PAY_3`, `PAY_4`, `PAY_5`, `PAY_6`  
  (repayment status for the last 6 months; higher values = more delayed)

**Billing history**
- `BILL_AMT1` … `BILL_AMT6` — monthly statement balances

**Payment history**
- `PAY_AMT1` … `PAY_AMT6` — monthly payments made


In [12]:
# =============================================================================
# DATA QUALITY CHECKS
# =============================================================================
section("Data quality checks")

print("Missing values per column:")
na = df.isna().sum().sort_values(ascending=False)
print(na[na>0].to_string() if (na>0).any() else "No missing values found.")

# Duplicate ID check 
if "ID" in df.columns:
    dup = df["ID"].duplicated().sum()
    print(f"Duplicate IDs: {dup}")

section("describe() — numeric")
print(df.describe().T.to_string())



Data quality checks
Missing values per column:
No missing values found.
Duplicate IDs: 0

describe() — numeric
                      count           mean            std       min       25%       50%        75%        max
ID                  30000.0   15000.500000    8660.398374       1.0   7500.75   15000.5   22500.25    30000.0
LIMIT_BAL           30000.0  167484.322667  129747.661567   10000.0  50000.00  140000.0  240000.00  1000000.0
SEX                 30000.0       1.603733       0.489129       1.0      1.00       2.0       2.00        2.0
EDUCATION           30000.0       1.853133       0.790349       0.0      1.00       2.0       2.00        6.0
MARRIAGE            30000.0       1.551867       0.521970       0.0      1.00       2.0       2.00        3.0
AGE                 30000.0      35.485500       9.217904      21.0     28.00      34.0      41.00       79.0
PAY_0               30000.0      -0.016700       1.123802      -2.0     -1.00       0.0       0.00        8.0
PAY_2   

In [ ]:
# checking class imbalance

# =============================================================================
# TARGET DISTRIBUTION / CLASS IMBALANCE
# =============================================================================
section("Target distribution (class imbalance)")
random_colors = random.sample(px.colors.qualitative.Plotly, k=len(vc_df))

target = "DEFAULT_NEXT_MONTH"
vc = df[target].value_counts(dropna=False).sort_index()
rate = df[target].mean()

print("Target counts:")
print(vc.to_string())
print(f"Default rate = {rate:.4f} ({rate*100:.2f}%)")

vc_df = vc.reset_index()
vc_df.columns = [target, "count"]
fig = px.bar(
    vc_df.assign(**{target: vc_df[target].astype(str)}), 
    x=target, 
    y="count", 
    title="Target distribution (0=no default, 1=default)",
    color=target,                              
    color_discrete_map={"0": "#2E86AB", "1": "#D1495B"})
fig.update_layout(
    title_x=0.02,
    xaxis_title=target,
    yaxis_title="count",
    legend_title_text="Default"
)
pio.templates.default = "plotly_white"
fig.show()



Target distribution (class imbalance)
Target counts:
DEFAULT_NEXT_MONTH
0    23364
1     6636
Default rate = 0.2212 (22.12%)


In [ ]:
# checking categorical variable distributions
# =============================================================================
# CATEGORICAL VARIABLES — DISTRIBUTIONS
# =============================================================================
section("Categorical variables distributions")

cat_cols = ["SEX", "EDUCATION", "MARRIAGE"]
for c in ["SEX", "EDUCATION", "MARRIAGE"]:
    print("\n" + "-" * 80)
    print(f"{c} value_counts:")
    print(df[c].value_counts(dropna=False).to_string())

    vc = df[c].value_counts(dropna=False).reset_index()
    vc.columns = [c, "count"]

    fig = px.bar(
        vc,
        x=c, y="count",
        title=f"{c} distribution",
        labels={c: c, "count": "count"}
    )
    fig.update_traces(marker_color=px.colors.qualitative.Set2)
    pio.templates.default = "plotly_white"
    fig.show()




Categorical variables distributions

--------------------------------------------------------------------------------
SEX value_counts:
SEX
2    18112
1    11888



--------------------------------------------------------------------------------
EDUCATION value_counts:
EDUCATION
2    14030
1    10585
3     4917
5      280
4      123
6       51
0       14



--------------------------------------------------------------------------------
MARRIAGE value_counts:
MARRIAGE
2    15964
1    13659
3      323
0       54


In [ ]:
# checking data quality issues in EDUCATION column for values 0 and 6 which are not defined in the original dataset description
df[(df["EDUCATION"] == 0)]
#df[(df["EDUCATION"] == 6)]

,ID,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,PAY_5,PAY_6,BILL_AMT1,BILL_AMT2,BILL_AMT3,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,DEFAULT_NEXT_MONTH,PAY_0_BIN
3769,3770,290000,2,0,2,38,1,-1,-1,-1,-1,-1,0,1437,3070,1406,2196,1481,1437,3078,1406,2196,1481,0,0,1
5945,5946,270000,1,0,2,39,1,-1,-1,-1,-1,-2,0,10193,69553,18607,0,0,10193,70213,19008,399,0,0,0,1
6876,6877,360000,1,0,2,30,0,0,-1,0,0,-1,40250,23022,12272,34345,36777,30,23000,12280,25007,25008,1767,3300,0,0
14631,14632,350000,2,0,2,53,-1,-1,-1,-1,-1,-1,5095,4815,61044,22611,1385,6043,4840,61349,22687,1389,6058,1153,0,-1
15107,15108,210000,1,0,2,45,-2,-2,-2,-2,-2,-2,2563,5854,1032,788,3499,3372,5854,1032,788,3565,3372,15381,0,-2
16881,16882,100000,1,0,2,37,0,0,-2,-2,-2,-2,7642,0,0,0,0,0,0,0,0,0,0,0,0,0
16896,16897,200000,1,0,2,40,1,-2,-1,-1,-1,-2,0,0,200,1000,0,0,0,200,1000,0,0,0,0,1
17414,17415,230000,2,0,2,47,-1,-1,-1,2,-1,-1,8394,5743,1336,255,5425,4838,5743,1598,0,5425,4838,3840,0,-1
19920,19921,50000,2,0,1,40,0,0,0,0,0,0,44749,46229,46798,47647,40500,41921,2229,2298,2100,2500,1921,8432,0,0
20030,20031,200000,2,0,2,30,-1,-1,2,-1,-1,-1,17160,7289,2868,9470,5816,7809,2880,0,9470,5834,7809,2886,0,-1


In [ ]:
# checking numeric variable distributions and outliers with histograms and boxplots
# =============================================================================
# NUMERIC VARIABLES — DISTRIBUTIONS
# =============================================================================
section("Numeric variables distributions")

num_cols = ["LIMIT_BAL", "AGE"]
colors = ["steelblue", "indianred"]
for c, col in zip(num_cols, colors):
    fig = px.histogram(df, x=c, nbins=50, title=f"{c} — histogram")
    fig.update_traces(marker_color=col)
    fig.update_layout(bargap=0.15)
    fig.show()

fig = px.box(df, y="LIMIT_BAL", title="LIMIT_BAL — boxplot")
fig.update_traces(marker_color="steelblue")       # <- only one color line
fig.show()

fig = px.box(df, y="AGE", title="AGE — boxplot")
fig.update_traces(marker_color="indianred")       # <- only one color line
fig.show()



Numeric variables distributions


Investigating limit for customers over 0.5m, there are some that default it in payment and the data looks correct.

In [49]:
# checking outliers if they are real or data errors
high_limit1 = df[df["LIMIT_BAL"] > 800000]
high_limit2 = df[df["LIMIT_BAL"] > 500000]

high_limit1

,ID,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,PAY_5,PAY_6,BILL_AMT1,BILL_AMT2,BILL_AMT3,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,DEFAULT_NEXT_MONTH,PAY_0_BIN
2197,2198,1000000,2,1,1,47,0,0,0,-1,0,0,964511,983931,535020,891586,927171,961664,50784,50723,896040,50000,50000,50256,0,0


In [50]:
high_limit2

,ID,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,PAY_5,PAY_6,BILL_AMT1,BILL_AMT2,BILL_AMT3,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,DEFAULT_NEXT_MONTH,PAY_0_BIN
12,13,630000,2,2,2,41,-1,0,-1,-1,-1,-1,12137,6500,6500,6500,6500,2870,1000,6500,6500,6500,2870,0,0,-1
260,261,510000,2,1,2,29,0,0,0,0,0,0,78331,99414,107686,103776,87265,36739,40010,20094,5000,5001,25365,65000,0,0
433,434,580000,2,1,1,36,0,0,0,0,0,0,159760,162189,166127,169365,168755,167964,6422,6565,5951,6006,5894,5946,0,0
451,452,600000,1,1,1,53,2,2,0,0,0,0,467150,458862,469703,447130,440982,434715,0,18000,16000,16000,21000,20000,1,2
527,528,620000,2,2,1,45,2,2,0,0,0,0,160837,156839,160440,163781,167159,170894,0,6200,6000,6000,6500,6000,1,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
29571,29572,570000,1,1,2,33,0,0,0,0,0,-2,388897,253793,261176,266800,0,0,9083,11472,12000,0,0,0,0,0
29740,29741,620000,1,2,2,31,-2,-2,-2,-2,-2,-2,5712,11598,21049,13846,3565,7076,11881,21171,13915,3583,7111,1971,0,-2
29861,29862,650000,1,1,1,44,-2,-2,-2,-2,-2,-2,2119,5094,5158,7139,1034,2127,5115,5180,7201,1035,2139,3463,0,-2
29886,29887,630000,1,2,1,46,0,0,0,0,0,0,125975,91247,81317,146005,146207,106467,3416,4300,84700,4211,4470,3600,0,0


this customer is real outlier 
LIMIT_BAL = 1,000,000
BILL_AMT ~ 900k–980k in several months
Some PAY_AMT are small (50k)
One PAY_AMT3 = 896,040 (very large)
but still fit into the repayments - no delay history

In [ ]:
# =============================================================================
# TARGET vs FEATURES 
# =============================================================================
# checking default rate by categories of key features 
# calculating default rate by categories of key features to identify which features are most predictive 
# of the target variable and to gain insights into the relationships between features and default risk. 

section("Target rate by categories")

pay_cols = ["PAY_0","PAY_2","PAY_3","PAY_4","PAY_5","PAY_6"]

for c in pay_cols:
    print("\n" + "-"*80)
    print(f"{c} value_counts (top):")
    print(df[c].value_counts(dropna=False).head(12).to_string())

    g = df.groupby(c)["DEFAULT_NEXT_MONTH"].agg(count="size", default_rate="mean").reset_index()
    g["default_rate_pct"] = (g["default_rate"] * 100).round(2)
    g = g.sort_values("default_rate_pct", ascending=False)

    print(f"\nDefault rate by {c}:")
    print(g.to_string(index=False))

    fig = px.bar(
    g.sort_values(c),
    x=c,
    y="default_rate_pct",
    color=c, 
    title=f"Default rate (%) by {c}",
    labels={"default_rate_pct": "default rate (%)"},
    template="plotly_white"
    )
    pio.templates.default = "plotly_white"
    fig.show()




Target rate by categories

--------------------------------------------------------------------------------
PAY_0 value_counts (top):
PAY_0
 0    14737
-1     5686
 1     3688
-2     2759
 2     2667
 3      322
 4       76
 5       26
 8       19
 6       11
 7        9

Default rate by PAY_0:
 PAY_0  count  default_rate  default_rate_pct
     7      9      0.777778             77.78
     3    322      0.757764             75.78
     2   2667      0.691414             69.14
     4     76      0.684211             68.42
     8     19      0.578947             57.89
     6     11      0.545455             54.55
     5     26      0.500000             50.00
     1   3688      0.339479             33.95
    -1   5686      0.167781             16.78
    -2   2759      0.132294             13.23
     0  14737      0.128113             12.81



--------------------------------------------------------------------------------
PAY_2 value_counts (top):
PAY_2
 0    15730
-1     6050
 2     3927
-2     3782
 3      326
 4       99
 1       28
 5       25
 7       20
 6       12
 8        1

Default rate by PAY_2:
 PAY_2  count  default_rate  default_rate_pct
     6     12      0.750000             75.00
     3    326      0.616564             61.66
     5     25      0.600000             60.00
     7     20      0.600000             60.00
     2   3927      0.556150             55.61
     4     99      0.505051             50.51
    -2   3782      0.182708             18.27
     1     28      0.178571             17.86
    -1   6050      0.159669             15.97
     0  15730      0.159123             15.91
     8      1      0.000000              0.00



--------------------------------------------------------------------------------
PAY_3 value_counts (top):
PAY_3
 0    15764
-1     5938
-2     4085
 2     3819
 3      240
 4       76
 7       27
 6       23
 5       21
 1        4
 8        3

Default rate by PAY_3:
 PAY_3  count  default_rate  default_rate_pct
     7     27      0.814815             81.48
     8      3      0.666667             66.67
     6     23      0.608696             60.87
     4     76      0.578947             57.89
     3    240      0.575000             57.50
     5     21      0.571429             57.14
     2   3819      0.515580             51.56
     1      4      0.250000             25.00
    -2   4085      0.185312             18.53
     0  15764      0.174512             17.45
    -1   5938      0.155945             15.59



--------------------------------------------------------------------------------
PAY_4 value_counts (top):
PAY_4
 0    16455
-1     5687
-2     4348
 2     3159
 3      180
 4       69
 7       58
 5       35
 6        5
 1        2
 8        2

Default rate by PAY_4:
 PAY_4  count  default_rate  default_rate_pct
     7     58      0.827586             82.76
     4     69      0.666667             66.67
     3    180      0.611111             61.11
     2   3159      0.523267             52.33
     5     35      0.514286             51.43
     8      2      0.500000             50.00
     1      2      0.500000             50.00
     6      5      0.400000             40.00
    -2   4348      0.192502             19.25
     0  16455      0.183288             18.33
    -1   5687      0.158959             15.90



--------------------------------------------------------------------------------
PAY_5 value_counts (top):
PAY_5
 0    16947
-1     5539
-2     4546
 2     2626
 3      178
 4       84
 7       58
 5       17
 6        4
 8        1

Default rate by PAY_5:
 PAY_5  count  default_rate  default_rate_pct
     8      1      1.000000            100.00
     7     58      0.827586             82.76
     6      4      0.750000             75.00
     3    178      0.634831             63.48
     4     84      0.607143             60.71
     5     17      0.588235             58.82
     2   2626      0.541889             54.19
    -2   4546      0.196876             19.69
     0  16947      0.188529             18.85
    -1   5539      0.161943             16.19



--------------------------------------------------------------------------------
PAY_6 value_counts (top):
PAY_6
 0    16286
-1     5740
-2     4895
 2     2766
 3      184
 4       49
 7       46
 6       19
 5       13
 8        2

Default rate by PAY_6:
 PAY_6  count  default_rate  default_rate_pct
     8      2      1.000000            100.00
     7     46      0.826087             82.61
     6     19      0.736842             73.68
     3    184      0.641304             64.13
     4     49      0.632653             63.27
     5     13      0.538462             53.85
     2   2766      0.506508             50.65
    -2   4895      0.200409             20.04
     0  16286      0.188444             18.84
    -1   5740      0.169861             16.99


In [ ]:
# c reating a summary table for PAY_0 vs DEFAULT_NEXT_MONTH to show the default rate 
# by categories of PAY_0 feature which represents the repayment status in the previous month 
# and is likely to be a strong predictor of default risk. This table will help us understand 
# how default rates vary across different repayment statuses and identify which categories of PAY_0 are associated with higher default risk.

TARGET = "DEFAULT_NEXT_MONTH"

g = (
    df.groupby("PAY_0")[TARGET]
      .agg(count="size", default_rate="mean")
      .reset_index()
)

g["default_rate_pct"] = (g["default_rate"] * 100).round(2)

g = g.sort_values("PAY_0")

# Clean final table
table = g[["PAY_0", "count", "default_rate_pct"]]
table.columns = ["PAY_0 (Months Past Due)", "Number of Customers", "Default Rate (%)"]

styled_table = (
    table.style
    .background_gradient(subset=["Default Rate (%)"], cmap="Reds")
    .set_properties(**{
        "background-color": "white",
        "color": "black",
        "border-color": "black"
    })
)

styled_table

,PAY_0 (Months Past Due),Number of Customers,Default Rate (%)
0,-2,2759,13.230000
1,-1,5686,16.780000
2,0,14737,12.810000
3,1,3688,33.950000
4,2,2667,69.140000
5,3,322,75.780000
6,4,76,68.420000
7,5,26,50.000000
8,6,11,54.550000
9,7,9,77.780000


In [17]:
# PAY_0 has a wide range of values, bin it for better analysis
# putting together everything over the repayment status in the first month (PAY_0) into bins: -2, -1, 0, 1, 2, 3, 4 
# to remove the imbalance and make it easier to analyse. We can treat all values of 4 or higher as a single category "4+" since they represent increasingly severe delinquency.

df["PAY_0_BIN"] = df["PAY_0"].clip(lower=-2, upper=4)
#df["PAY_0_BIN"] = df["PAY_0_BIN"].replace({4: "4+"})

In [18]:
print_df(df, "cleaned")


-----------------------------------------------------------------------------------------------
[DATAFRAME] cleaned | shape = 30,000 rows × 26 cols
-----------------------------------------------------------------------------------------------
   ID  LIMIT_BAL  SEX  EDUCATION  MARRIAGE  AGE  PAY_0  PAY_2  PAY_3  PAY_4  PAY_5  PAY_6  BILL_AMT1  BILL_AMT2  BILL_AMT3  BILL_AMT4  BILL_AMT5  BILL_AMT6  PAY_AMT1  PAY_AMT2  PAY_AMT3  PAY_AMT4  PAY_AMT5  PAY_AMT6  DEFAULT_NEXT_MONTH  PAY_0_BIN
0   1      20000    2          2         1   24      2      2     -1     -1     -2     -2       3913       3102        689          0          0          0         0       689         0         0         0         0                   1          2
1   2     120000    2          2         2   26     -1      2      0      0      0      2       2682       1725       2682       3272       3455       3261         0      1000      1000      1000         0      2000                   1         -1
2   3      900

In [ ]:
# checking the distinct values in the new binned PAY_0_BIN column to confirm the binning worked as expected
sorted(df["PAY_0_BIN"].unique())

[np.int64(-2),
 np.int64(-1),
 np.int64(0),
 np.int64(1),
 np.int64(2),
 np.int64(3),
 np.int64(4)]

In [81]:

# =============================================================================
# ANALYSIS: PAY_0 
# =============================================================================
# PAY_0 is the repayment status in the first month (the most recent month). It has a wide range of values. 
# Creating code to analyze the distribution of PAY_0 and its relationship with default rate, both with exact values and binned.

section("PAY_0 analysis")
colors = (
    px.colors.qualitative.Set3 +
    px.colors.qualitative.Dark24
)
pay_col = "PAY_0"

print("\n" + "-" * 80)
print(f"{pay_col} value_counts (top):")
print(df[pay_col].value_counts(dropna=False).head(15).to_string())

# 2) Default rate by exact PAY_0
g_exact = (
    df.groupby(pay_col)["DEFAULT_NEXT_MONTH"]
      .agg(count="size", default_rate="mean")
      .reset_index()
)
g_exact["default_rate_pct"] = (g_exact["default_rate"] * 100).round(2)
g_exact = g_exact.sort_values("default_rate_pct", ascending=False)

print("\nDefault rate by PAY_0 (exact):")
print(g_exact.to_string(index=False))

fig = px.bar(
    g_exact.sort_values(pay_col),
    x=pay_col,
    y="default_rate_pct",
    color=pay_col,
    color_discrete_sequence=colors,
    title="Default rate (%) by PAY_0 (exact values)",
    labels={"default_rate_pct": "default rate (%)"}
)
pio.templates.default = "plotly_white"
fig.show()

df["PAY_0_BIN"] = df[pay_col].clip(lower=-2, upper=4)
#df["PAY_0_BIN"] = df["PAY_0_BIN"].replace({4: "4"})

print("\nPAY_0_BIN value_counts:")
print(df["PAY_0_BIN"].value_counts(dropna=False).to_string())

g_bin = (
    df.groupby("PAY_0_BIN")["DEFAULT_NEXT_MONTH"]
      .agg(count="size", default_rate="mean")
      .reset_index()
)
g_bin["default_rate_pct"] = (g_bin["default_rate"] * 100).round(2)
g_bin = g_bin.sort_values("default_rate_pct", ascending=False)

print("\nDefault rate by PAY_0_BIN:")
print(g_bin.to_string(index=False))

fig = px.bar(
    g_bin,
    x="PAY_0_BIN", y="default_rate_pct",
    color="PAY_0_BIN",
    title="Default rate (%) by PAY_0_BIN (binned)",
    labels={"default_rate_pct": "default rate (%)"}
)
fig.update_traces(marker_color=px.colors.qualitative.Set2)
pio.templates.default = "plotly_white"
fig.show()

baseline = df["DEFAULT_NEXT_MONTH"].mean() * 100
print("\nBaseline default rate (%):", round(baseline, 2))

g_lift = g_bin.copy()
g_lift["lift_vs_baseline_x"] = (g_lift["default_rate_pct"] / baseline).round(2)
print("\nLift vs baseline (PAY_0_BIN):")
print(g_lift.sort_values("lift_vs_baseline_x", ascending=False).to_string(index=False))



PAY_0 analysis

--------------------------------------------------------------------------------
PAY_0 value_counts (top):
PAY_0
 0    14737
-1     5686
 1     3688
-2     2759
 2     2667
 3      322
 4       76
 5       26
 8       19
 6       11
 7        9

Default rate by PAY_0 (exact):
 PAY_0  count  default_rate  default_rate_pct
     7      9      0.777778             77.78
     3    322      0.757764             75.78
     2   2667      0.691414             69.14
     4     76      0.684211             68.42
     8     19      0.578947             57.89
     6     11      0.545455             54.55
     5     26      0.500000             50.00
     1   3688      0.339479             33.95
    -1   5686      0.167781             16.78
    -2   2759      0.132294             13.23
     0  14737      0.128113             12.81



PAY_0_BIN value_counts:
PAY_0_BIN
 0    14737
-1     5686
 1     3688
-2     2759
 2     2667
 3      322
 4      141

Default rate by PAY_0_BIN:
 PAY_0_BIN  count  default_rate  default_rate_pct
         3    322      0.757764             75.78
         2   2667      0.691414             69.14
         4    141      0.631206             63.12
         1   3688      0.339479             33.95
        -1   5686      0.167781             16.78
        -2   2759      0.132294             13.23
         0  14737      0.128113             12.81



Baseline default rate (%): 22.12

Lift vs baseline (PAY_0_BIN):
 PAY_0_BIN  count  default_rate  default_rate_pct  lift_vs_baseline_x
         3    322      0.757764             75.78                3.43
         2   2667      0.691414             69.14                3.13
         4    141      0.631206             63.12                2.85
         1   3688      0.339479             33.95                1.53
        -1   5686      0.167781             16.78                0.76
        -2   2759      0.132294             13.23                0.60
         0  14737      0.128113             12.81                0.58


Above default rate - 22% - means 22% missed the payment at least once. It is weighted average from all customers

Simple Interpretation

High Risk Groups

3 months late → 75.8% default rate
3.43× more likely to default than average
2 months late → 69.1% default rate
3.13× baseline risk
4+ months late → 63.1% default rate
2.85× baseline risk

extremely high-risk customers.

Medium Risk

1 month late → 33.9% default
1.53× baseline risk
Still higher than average.

Low Risk

On time (0) → 12.8% default
Only 0.58× baseline risk
Paid early (-1, -2) → 13–16%
0.60–0.76× baseline risk


In [82]:
# =============================================================================
# REPAYMENT STATUS 
# =============================================================================
section("Repayment status (PAY_*) overview")

pay_cols = ["PAY_0","PAY_2","PAY_3","PAY_4","PAY_5","PAY_6"]
for c in pay_cols:
    print("\n" + "-"*80)
    print(f"{c} value_counts (top):")
    print(df[c].value_counts().head(12).to_string())

# Default rate by each PAY_ * status
section("Default rate by PAY_0")
g = (df.groupby("PAY_0")["DEFAULT_NEXT_MONTH"]
       .agg(["count","mean"])
       .rename(columns={"mean":"default_rate"}))
g["default_rate_pct"] = (g["default_rate"]*100).round(2)
print(g.sort_values("default_rate", ascending=False).to_string())

fig = px.bar(g.reset_index(), x="PAY_0", y="default_rate_pct",
             title="Default rate (%) by PAY_0",
             labels={"default_rate_pct":"default rate (%)"},
             color="PAY_0",
             color_discrete_sequence=colors)
pio.templates.default = "plotly_white"
fig.show()



Repayment status (PAY_*) overview

--------------------------------------------------------------------------------
PAY_0 value_counts (top):
PAY_0
 0    14737
-1     5686
 1     3688
-2     2759
 2     2667
 3      322
 4       76
 5       26
 8       19
 6       11
 7        9

--------------------------------------------------------------------------------
PAY_2 value_counts (top):
PAY_2
 0    15730
-1     6050
 2     3927
-2     3782
 3      326
 4       99
 1       28
 5       25
 7       20
 6       12
 8        1

--------------------------------------------------------------------------------
PAY_3 value_counts (top):
PAY_3
 0    15764
-1     5938
-2     4085
 2     3819
 3      240
 4       76
 7       27
 6       23
 5       21
 1        4
 8        3

--------------------------------------------------------------------------------
PAY_4 value_counts (top):
PAY_4
 0    16455
-1     5687
-2     4348
 2     3159
 3      180
 4       69
 7       58
 5       35
 6        5
 1   

Distribution of Repayment Status (PAY_*)

Across PAY_0–PAY_6:

The majority of customers are PAY = 0 (paid on time)
A large group also have -1 or -2 (paid early / no delay)
Fewer customers are 2+ months late
Very small numbers are 5–8 months late

This shows:

Most customers are low risk, but a small delinquent group carries very high risk.

Default Rate by PAY_0 (Most Recent Month)

Baseline default rate: 22.12%

Groups:

1. Low Risk
PAY_0	Default Rate
0 (on time)	12.81%
-1	16.78%
-2	13.23%

2. Medium Risk
PAY_0	Default Rate
1 month late	33.95%

About 1.5× baseline risk.

3. Very High Risk
PAY_0	Default Rate
2 months late	69.14%
3 months late	75.78%
4 months late	68.42%
5+ months late	50–78%

These customers are:

3× more likely to default than average.

Key Insight

Default probability increases dramatically as delinquency increases.
The relationship is:

Strong
Monotonic (generally increasing)
Non-linear
Extremely predictive

This confirms that PAY_0 is a dominant risk variable.

In [65]:
# =============================================================================
# BILLING / PAYMENTS — MONTHLY PATTERNS (mean / median)
# =============================================================================
# calculating mean and median for billing amounts and payment amounts over the last 6 months to 
# understand typical values and potential outliers, and how they evolve over time.

section("Billing and payments — monthly patterns")

bill_cols = [f"BILL_AMT{i}" for i in range(1,7)]
pay_amt_cols = [f"PAY_AMT{i}" for i in range(1,7)]

bill_stats = df[bill_cols].agg(["mean","median"]).T
pay_stats  = df[pay_amt_cols].agg(["mean","median"]).T

print("\nBilling stats (mean/median):")
print(bill_stats.to_string())
print("\nPayment stats (mean/median):")
print(pay_stats.to_string())

# Line charts
fig = go.Figure()
fig.add_trace(go.Scatter(x=bill_stats.index, y=bill_stats["mean"], mode="lines+markers", name="Bill mean"))
fig.add_trace(go.Scatter(x=bill_stats.index, y=bill_stats["median"], mode="lines+markers", name="Bill median"))
fig.update_layout(title="Billing amounts over last 6 months (mean vs median)", xaxis_title="month column", yaxis_title="amount")
pio.templates.default = "plotly_white"
fig.show()

fig = go.Figure()
fig.add_trace(go.Scatter(x=pay_stats.index, y=pay_stats["mean"], mode="lines+markers", name="Pay mean"))
fig.add_trace(go.Scatter(x=pay_stats.index, y=pay_stats["median"], mode="lines+markers", name="Pay median"))
fig.update_layout(title="Payment amounts over last 6 months (mean vs median)", xaxis_title="month column", yaxis_title="amount")
pio.templates.default = "plotly_white"
fig.show()



Billing and payments — monthly patterns

Billing stats (mean/median):
                   mean   median
BILL_AMT1  51223.330900  22381.5
BILL_AMT2  49179.075167  21200.0
BILL_AMT3  47013.154800  20088.5
BILL_AMT4  43262.948967  19052.0
BILL_AMT5  40311.400967  18104.5
BILL_AMT6  38871.760400  17071.0

Payment stats (mean/median):
                 mean  median
PAY_AMT1  5663.580500  2100.0
PAY_AMT2  5921.163500  2009.0
PAY_AMT3  5225.681500  1800.0
PAY_AMT4  4826.076867  1500.0
PAY_AMT5  4799.387633  1500.0
PAY_AMT6  5215.502567  1500.0


Billing Amounts (BILL_AMT1–6)

* **Mean > Median for all months**
* Bills gradually decrease from Month 1 to Month 6

Example:

 Month      Mean Bill  Median Bill 
 BILL_AMT1  51,223     22,381      
 BILL_AMT6  38,872     17,071      

**Strong right skew**

Since:
Mean >> Median
spending distribution is highly skewed.

**Bills trend downward slightly over 6 months**

The average bill decreases over time:

51k → 38k

This trend feature could be useful for:

* Risk slope features
* Behaviour change detection

Payment Amounts (PAY_AMT1–6)

### Key observations:

Month    | Mean Payment | Median Payment 

PAY_AMT1 | 5,663        | 2,100          
PAY_AMT6 | 5,215        | 1,500          

Again:

Mean >> Median

Strong skew.

Bills are large, skewed.
Payments are smaller and also skewed.


In [83]:

# =============================================================================
# CORRELATIONS 
# =============================================================================

# checking correlations between numeric features for Pearson and Spearman methods, 
# and quick top correlations with target to identify which features have the strongest linear and monotonic relationships with the target variable.

section("Correlation heatmap")

# Correlation can be dominated by outliers; also checking Spearman in addition to Pearson.
num_for_corr = df.drop(columns=["ID"], errors="ignore").select_dtypes(include=[np.number]).copy()

corr_pearson = num_for_corr.corr(method="pearson")
corr_spearman = num_for_corr.corr(method="spearman")

fig = px.imshow(corr_pearson, title="Correlation heatmap- Pearson", aspect="auto")
fig.show()

fig = px.imshow(corr_spearman, title="Correlation heatmap- Spearman", aspect="auto")
fig.show()

section("Top absolute correlations with target - Pearson")
target_corr = corr_pearson["DEFAULT_NEXT_MONTH"].drop("DEFAULT_NEXT_MONTH").abs().sort_values(ascending=False)
print(target_corr.head(20).to_string())

section("Top absolute correlations with target - Spearman")
target_corr = corr_spearman["DEFAULT_NEXT_MONTH"].drop("DEFAULT_NEXT_MONTH").abs().sort_values(ascending=False)
print(target_corr.head(20).to_string())



Correlation heatmap



Top absolute correlations with target - Pearson
PAY_0_BIN    0.328055
PAY_0        0.324794
PAY_2        0.263551
PAY_3        0.235253
PAY_4        0.216614
PAY_5        0.204149
PAY_6        0.186866
LIMIT_BAL    0.153520
PAY_AMT1     0.072929
PAY_AMT2     0.058579
PAY_AMT4     0.056827
PAY_AMT3     0.056250
PAY_AMT5     0.055124
PAY_AMT6     0.053183
SEX          0.039961
EDUCATION    0.028006
MARRIAGE     0.024339
BILL_AMT1    0.019644
BILL_AMT2    0.014193
BILL_AMT3    0.014076

Top absolute correlations with target - Spearman
PAY_0_BIN    0.292215
PAY_0        0.292213
PAY_2        0.216919
PAY_3        0.194771
PAY_4        0.173690
LIMIT_BAL    0.169586
PAY_AMT1     0.160493
PAY_5        0.159043
PAY_AMT2     0.150977
PAY_6        0.142523
PAY_AMT3     0.139388
PAY_AMT4     0.127979
PAY_AMT6     0.121444
PAY_AMT5     0.116587
EDUCATION    0.044369
SEX          0.039961
MARRIAGE     0.026490
BILL_AMT1    0.025327
BILL_AMT2    0.015554
BILL_AMT3    0.012670


These are **Pearson correlations** between each variable and the target:

`DEFAULT_NEXT_MONTH`

Strongest Predictors

Top correlations:

 PAY_0 - **0.325**   
 PAY_2 - 0.264       
 PAY_3 - 0.235       
 PAY_4 - 0.217       
 PAY_5 - 0.204       
 PAY_6 - 0.187       

### Interpretation:

PAY_0 alone has correlation ≈ 0.325 — which is very high for behavioural financial data.

The correlation decreases as we go further back in time (PAY_6).
More recent behaviour matters more.

# Moderate Predictors

 LIMIT_BAL- 0.154  
Credit limit has moderate correlation.

Pearson measures **linear relationship only**.

Tree-based models (LightGBM/XGBoost):

> Pearson correlation analysis indicates that repayment status variables (PAY_0–PAY_6) exhibit the strongest linear association with default, with PAY_0 showing the highest correlation (r = 0.325). However, given the ordinal nature of repayment variables and the presence of skewness in financial features, Spearman correlation provides a more appropriate measure of monotonic association. Both analyses confirm the dominant role of recent repayment behaviour in predicting default risk.

These are **Spearman correlations**
Strongest Predictors 

# Top variables:

PAY_0_BIN → 0.292
PAY_0 → 0.292
PAY_2 → 0.217
PAY_3 → 0.195
PAY_4 → 0.174

# Interpretation

Repayment status variables remain the strongest predictors.
The monotonic association confirms that higher repayment delay corresponds to higher default likelihood.
Correlation strength decreases for older repayment months → recent behaviour matters most.

# Changes:

Payment amounts (PAY_AMT1–6)

Under Pearson → ~0.05–0.07 (weak)
Under Spearman → 0.12–0.16 (moderate)

Spearman captures monotonic patterns even if not linear.


In [ ]:
# =============================================================================
# FEATURE ENGINEERING 
# =============================================================================

# in here we are trying some common feature engineering ideas for this dataset, especially around the billing and payment history which is often key for credit default prediction. 
# This includes creating payment ratio features, utilisation proxies, aggregates over time, and delinquency features based on the PAY_X columns.

section("Feature engineering")

df_fe = df.copy()
# cleaning and recoding categorical variables based on the original dataset description, some values in EDUCATION and MARRIAGE columns are not defined 
# in the original dataset description, grouping them into an "Other" category (4 for EDUCATION and 3 for MARRIAGE) 

# EDUCATION: 
df_fe["EDUCATION"] = df_fe["EDUCATION"].replace({0: 4, 5: 4, 6: 4})
# MARRIAGE: 
df_fe["MARRIAGE"] = df_fe["MARRIAGE"].replace({0: 3})

#Payment ratio features 
bill_cols = [f"BILL_AMT{i}" for i in range(1,7)]
pay_cols_amt = [f"PAY_AMT{i}" for i in range(1,7)]

for i in range(1,7):
    b = f"BILL_AMT{i}"
    p = f"PAY_AMT{i}"
    df_fe[f"PAY_RATIO{i}"] = df_fe[p] / (df_fe[b].abs() + 1)  # +1 to avoid division by zero

#  Utilization proxy
# - clip(lower=0) avoids negative bill artifacts; negative bills can reflect credits.
# - mean(axis=1) computes average bill across the 6 months for each customer (row-wise).
# - divide by LIMIT_BAL (+1 prevents division by zero)

df_fe["UTILIZATION6M_MEAN"] = (df_fe[bill_cols].clip(lower=0).mean(axis=1)) / (df_fe["LIMIT_BAL"] + 1)

# Aggregates over 6 months
# - BILL_TOTAL_6M = total billed amount across 6 months
# - PAY_TOTAL_6M  = total payments across 6 months
# - PAY_RATIO_6M_MEAN = average monthly payment ratio across 6 months
df_fe["BILL_TOTAL_6M"] = df_fe[bill_cols].sum(axis=1)
df_fe["PAY_TOTAL_6M"]  = df_fe[pay_cols_amt].sum(axis=1)
df_fe["PAY_RATIO_6M_MEAN"] = df_fe[[f"PAY_RATIO{i}" for i in range(1,7)]].mean(axis=1)